# Approach 2: a set-prediction transformer (RT-DETR-L)

ENPM818Z, Lecture 4. RT-DETR-L is GP2's second model: a CNN backbone, a transformer
encoder, and a decoder with 300 **object queries**. Each query gives one answer, and
training pairs each real object with exactly one query, so there are no duplicates and
**no NMS**. This notebook follows the slides on the same photo.

1. Run the detector.
2. Open its raw output: one matrix, one row per query.
3. Check that the kept boxes need no NMS.
4. Pair queries with objects the way training does: the Hungarian algorithm.
5. Compute attention by hand, with the numbers from the slides.
6. Compare with YOLOv8s.

**Requirements:** `pip install ultralytics` (PyTorch, NumPy, matplotlib) and SciPy
(`pip install scipy`) for section 4. A GPU is optional. The weights (`rtdetr-l.pt`,
66 MB) download on the first run.

In [ ]:
# This original version of the code is from fall 2025 and it was modified and
# improved by Anthropic Claude Opus 5.5.
# Signed: Zeid Kootbally

import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import ultralytics

# The photo from the slides. To use a CARLA frame from GP1, put its path here.
IMAGE = os.path.join(os.path.dirname(ultralytics.__file__), "assets", "bus.jpg")
photo = np.asarray(Image.open(IMAGE).convert("RGB"))
H, W = photo.shape[:2]
print(f"image: {W} x {H} pixels (width x height)")


def draw(ax, image, boxes, names, title):
    """boxes: list of ((x0, y0, x1, y1), class_id, score) in image pixels."""
    ax.imshow(image)
    ax.set_axis_off()
    ax.set_title(title)
    for (x0, y0, x1, y1), c, s in boxes:
        ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=2,
                                        ec="tab:orange"))
        ax.text(x0, y0, f"{names[c]} {s:.2f}", color="white", fontsize=8, va="bottom",
                bbox=dict(fc="tab:orange", ec="none", pad=1))


def iou(a, b):
    """Intersection over union of two boxes given as (x0, y0, x1, y1)."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter)


def latency_ms(model, n=50, warmup=10):
    """Median time of the whole predict call (file in, boxes out), batch 1."""
    for _ in range(warmup):
        model(IMAGE, verbose=False)
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        model(IMAGE, verbose=False)
        times.append(1000 * (time.perf_counter() - t0))
    return float(np.median(times))

from ultralytics import RTDETR

model = RTDETR("rtdetr-l.pt")      # COCO-pretrained, 80 classes

## 1. Run the detector

In [ ]:
result = model(IMAGE, verbose=False)[0]
final = [(b.tolist(), int(c), float(s))
         for b, c, s in zip(result.boxes.xyxy, result.boxes.cls, result.boxes.conf)]
for (x0, y0, x1, y1), c, s in final:
    print(f"{result.names[c]:>13s} {s:.3f}  ({x0:.1f}, {y0:.1f}) to ({x1:.1f}, {y1:.1f})")

fig, ax = plt.subplots(figsize=(5, 6.5))
draw(ax, photo, final, result.names, f"RT-DETR-L: {len(final)} objects")
plt.show()

## 2. The raw output: one matrix, one row per query

Ultralytics resizes the photo to 640 x 640 for RT-DETR. The output has **300 rows**, one
per query, and **84 columns**: the box center and size (cx, cy, w, h) as fractions of
the image width and height, then 80 class scores between 0 and 1 (a sigmoid). There is
no "no object" column: a query that found nothing has all 80 scores low.

In [ ]:
x = model.predictor.preprocess([photo[:, :, ::-1].copy()])
print("input tensor:", tuple(x.shape))
with torch.no_grad():
    out = model.predictor.model(x)
out = out[0] if isinstance(out, (list, tuple)) else out
queries = out[0].cpu().numpy()             # (300, 84)
print("output matrix:", queries.shape, " (one row per query, numbered from 1)")

best = queries[:, 4:].max(1)
order = np.argsort(-best)
print("the 10 most confident queries:")
for q in order[:10]:
    cx, cy, w, h = queries[q, :4]
    c = int(queries[q, 4:].argmax())
    print(f"  query {q + 1:3d}: {result.names[c]:>13s} {best[q]:.3f}   "
          f"center ({cx:.3f}, {cy:.3f}) = ({cx * W:.1f}, {cy * H:.1f}) pixels")
print(f"queries above 0.25: {(best > 0.25).sum()}, above 0.5: {(best > 0.5).sum()}")

## 3. No NMS: are there duplicates left?

If training really gave each object to one query, the kept boxes of the same class
should barely overlap. Measure the largest IoU between any two kept boxes of the same
class.

In [ ]:
largest = 0.0
for i in range(len(final)):
    for j in range(i + 1, len(final)):
        if final[i][1] == final[j][1]:
            largest = max(largest, iou(final[i][0], final[j][0]))
print(f"largest IoU between two kept boxes of the same class: {largest:.3f}")
print("NMS would only delete boxes above 0.7; there is nothing for it to do.")

## 4. Which query answers for which object? The Hungarian algorithm

During training, DETR pairs queries with the real objects one to one, with the lowest
total cost. The cost of a pair is low when the query gives that object's class a high
score and its box is close to the object's box. These are the numbers from the slide:
4 queries, 2 objects.

In [ ]:
from scipy.optimize import linear_sum_assignment

cost = np.array([[0.2, 0.3],      # rows: queries 1 to 4
                 [0.3, 0.9],      # columns: person, bus
                 [0.8, 0.8],
                 [0.9, 0.7]])
objects = ["person", "bus"]

rows, cols = linear_sum_assignment(cost)   # the Hungarian algorithm
for r, c in zip(rows, cols):
    print(f"query {r + 1} -> {objects[c]}  (cost {cost[r, c]})")
print(f"best total: {cost[rows, cols].sum():.1f}")

# greedy: take the cheapest pair first, then the cheapest pair left
used_q, total = set(), 0.0
for c in np.argsort(cost.min(0)):
    q = min((q for q in range(4) if q not in used_q), key=lambda q: cost[q, c])
    used_q.add(q)
    total += cost[q, c]
print(f"greedy total: {total:.1f}")

## 5. Attention by hand

The decoder's queries and the encoder's cells work with attention. Here is the wheel
example from the slides: three tokens, vectors of length d = 2, chosen for the example.
Each weight is a softmax of (query . key) / sqrt(d), and the new value is the weighted
average of the values.

In [ ]:
q = np.array([1.0, 0.5])                       # the wheel's query
K = np.array([[1.0, 0.0], [0.6, 0.8], [-0.5, 0.2]])   # keys: wheel, window, sky
V = np.array([[1.0, 0.0], [0.0, 1.0], [0.0, 0.0]])    # values
d = 2

scores = K @ q / np.sqrt(d)
weights = np.exp(scores) / np.exp(scores).sum()
print("q . k:          ", np.round(K @ q, 3))
print("divided by sqrt(2):", np.round(scores, 3))
print("weights:        ", np.round(weights, 3), " sum", round(weights.sum(), 3))
print("new value:      ", np.round(weights @ V, 3))

## 6. Side by side with YOLOv8s

The same photo, both GP2 models. Weights are counted as stored in each checkpoint. The
times are the whole predict call, median of 50 runs, on this machine; they will differ
from the slides on a different GPU.

In [ ]:
from ultralytics import YOLO

yolo = YOLO("yolov8s.pt")
for name, m, fresh in (("YOLOv8s", yolo, YOLO("yolov8s.pt")),
                       ("RT-DETR-L", model, RTDETR("rtdetr-l.pt"))):
    r = m(IMAGE, verbose=False)[0]
    # count on a freshly loaded model: after its first prediction Ultralytics folds
    # each batch-normalization layer into the convolution before it, which removes
    # weights, so a used model reports fewer than the checkpoint holds
    n_weights = sum(p.numel() for p in fresh.model.parameters())
    print(f"{name:>9s}: {len(r.boxes):2d} objects, {n_weights / 1e6:.1f} M weights, "
          f"{latency_ms(m):.1f} ms per image")

## Exercises

1. Set `IMAGE` to a camera frame from GP1. Compare the two models' objects and
   confidences. Which one reports more wrong objects above 0.25?
2. In section 2, print the 10 *least* confident queries. What do their scores tell you
   about the "no object" answer?
3. In section 4, change one cost so that greedy and Hungarian agree. What had to be
   true about the table?
4. In section 5, change the sky's key so that the wheel attends to it the most. Which
   key did you choose, and what is the new value?